# 03 – Indeks Leksikal: BM25 dan TF-IDF N-gram Karakter

Masukan: `chunks_L3_S2.parquet` (notebook 02) dan `lyrecall_text.py` (notebook 01).
Keluaran: indeks BM25 dan TF-IDF dalam bentuk *inverted index* (matriks CSC), siap dipakai notebook 06.

**BM25** (persamaan 1–2 laporan): token kata dari `text_lex`, k1 = 1,2, b = 0,75, IDF = ln(1 + (N − nt + 0,5)/(nt + 0,5)).

**TF-IDF n-gram karakter** (persamaan 3–4): n-gram karakter 3–5 dari `text_lex` (tanpa batas kata, sehingga kata yang
salah dengar atau tergabung/terpisah tetap berbagi n-gram), bobot tf · ln(N/df), kemiripan kosinus.
Fitur di-*hash* ke 2^22 dimensi (`HashingVectorizer`) agar tidak perlu kamus n-gram bermilyar entri dan bisa diparalelkan.

Nomor baris pada matriks = `chunk_id`.

In [1]:
# konfigurasi dan muat data
import os, sys, glob, gc, json, time
import numpy as np, pandas as pd, scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
from sklearn.preprocessing import normalize
from joblib import Parallel, delayed

sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-clean/lyrecall_text.py", recursive=True)[0]))
from lyrecall_text import normalize_lex

# ====== KONFIGURASI ======
N_LIMIT       = None
K1, B         = 1.2, 0.75    # BM25
NGRAM         = (3, 5)       # TF-IDF karakter
HASH_FEATURES = 2**22
SUBLINEAR_TF  = False        # False = tf mentah, sesuai persamaan (3)
OUT_DIR       = "/kaggle/working"

path = glob.glob("/kaggle/input/datasets/nshamid/lyrecall-chunks/chunks_L3_S2.parquet", recursive=True)[0]
chunks = pd.read_parquet(path, columns=["chunk_id", "song_id", "text_lex", "n_songs_same_text"])
if N_LIMIT:
    chunks = chunks.iloc[:N_LIMIT].reset_index(drop=True)
assert (chunks["chunk_id"].values == np.arange(len(chunks))).all(), "chunk_id harus berurutan"

texts   = chunks["text_lex"].tolist()
song_of = chunks["song_id"].values
N = len(texts)
print("Chunk:", f"{N:,}", "| lagu:", chunks["song_id"].nunique())

Chunk: 1,843,304 | lagu: 99821


## 1. Indeks BM25

In [2]:
t0 = time.time()
cv = CountVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, dtype=np.float32)
Xc = cv.fit_transform(texts)                      # chunk x kata (CSR)
dl = np.asarray(Xc.sum(axis=1)).ravel()
avgdl = float(dl.mean())
df_t = np.bincount(Xc.indices, minlength=Xc.shape[1])
idf = np.log1p((N - df_t + 0.5) / (df_t + 0.5)).astype(np.float32)

rows = np.repeat(np.arange(N), np.diff(Xc.indptr))
tf = Xc.data
Xc.data = (idf[Xc.indices] * tf * (K1 + 1) / (tf + K1 * (1 - B + B * dl[rows] / avgdl))).astype(np.float32)
bm25 = Xc.tocsc()                                 # kata x chunk: daftar posting per kata
del Xc, rows, tf; gc.collect()
vocab = cv.vocabulary_

print(f"BM25: {len(vocab):,} kata | rerata panjang chunk {avgdl:.1f} kata | "
      f"nnz {bm25.nnz:,} | {time.time()-t0:.0f} dtk")

BM25: 155,138 kata | rerata panjang chunk 20.8 kata | nnz 31,730,918 | 24 dtk


## 2. Indeks TF-IDF n-gram karakter
Langkah: hitung n-gram per batch secara paralel → hitung df → bobot tf·ln(N/df) → normalisasi L2 → simpan sebagai CSC.

In [3]:
t0 = time.time()
hv = HashingVectorizer(analyzer="char", ngram_range=NGRAM, n_features=HASH_FEATURES,
                       alternate_sign=False, norm=None, lowercase=False, dtype=np.float32)
def _tx(batch):
    return hv.transform(batch)

BATCH = 50_000
parts = Parallel(n_jobs=-1)(delayed(_tx)(texts[i:i + BATCH]) for i in range(0, N, BATCH))
X = sp.vstack(parts, format="csr"); del parts; gc.collect()
print(f"n-gram dihitung: nnz {X.nnz:,} | {time.time()-t0:.0f} dtk")

dfc = np.bincount(X.indices, minlength=HASH_FEATURES)
idf_c = np.log(N / np.maximum(dfc, 1)).astype(np.float32)
if SUBLINEAR_TF:
    X.data = 1 + np.log(X.data)
X.data *= idf_c[X.indices]
normalize(X, norm="l2", copy=False)

tfidf = X.tocsc()                                 # n-gram x chunk
del X; gc.collect()
print(f"TF-IDF siap: {tfidf.shape} | nnz {tfidf.nnz:,} | {time.time()-t0:.0f} dtk")

n-gram dihitung: nnz 462,631,841 | 134 dtk
TF-IDF siap: (1843304, 4194304) | nnz 462,631,841 | 155 dtk


## 3. Fungsi pencarian
Skor dihitung dengan menjumlahkan daftar posting dari kata atau n-gram kueri (tanpa menyalin matriks).
Kueri dinormalisasi dengan `normalize_lex` yang sama dengan indeks.

In [4]:
def score_postings(M, ids, weights, n_docs):
    s = np.zeros(n_docs, dtype=np.float32)
    ip, ix, dt = M.indptr, M.indices, M.data
    for j, w in zip(ids, weights):
        a, e = ip[j], ip[j + 1]
        s[ix[a:e]] += w * dt[a:e]
    return s

def topk(scores, k):
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    keep = scores[idx] > 0
    return idx[keep], scores[idx][keep]

EMPTY = (np.array([], dtype=int), np.array([], dtype=np.float32))

def search_bm25(q, k=100):
    ids = [vocab[t] for t in set(normalize_lex(q).split()) if t in vocab]
    if not ids:
        return EMPTY
    return topk(score_postings(bm25, ids, np.ones(len(ids), np.float32), N), k)

def search_tfidf(q, k=100):
    qv = hv.transform([normalize_lex(q)])
    ids = qv.indices
    if len(ids) == 0:
        return EMPTY
    w = qv.data.astype(np.float32)
    if SUBLINEAR_TF:
        w = 1 + np.log(w)
    w = w * idf_c[ids]
    n = np.linalg.norm(w)
    if n == 0:
        return EMPTY
    return topk(score_postings(tfidf, ids, w / n, N), k)

# contoh
i = 123
q = " ".join(texts[i].split()[:6])
for name, fn in [("BM25", search_bm25), ("TF-IDF", search_tfidf)]:
    idx, sc = fn(q, 3)
    print(name, "| kueri:", q)
    for c, s in zip(idx, sc):
        print(f"   chunk {c} (lagu {song_of[c]}) skor {s:.3f}: {texts[c][:70]}")

BM25 | kueri: oh theres the cheat in the
   chunk 122 (lagu 6) skor 19.707: the people are so great but really theres only me and that means im so
   chunk 123 (lagu 6) skor 18.547: oh theres the cheat in the place where tropical breezes flow the cheat
   chunk 1015093 (lagu 55100) skor 18.219: oh she fill the sheet sheet sheet oh she finna cheat cheat cheat oh sh
TF-IDF | kueri: oh theres the cheat in the
   chunk 122 (lagu 6) skor 0.499: the people are so great but really theres only me and that means im so
   chunk 846538 (lagu 45940) skor 0.483: theres theres theres
   chunk 123 (lagu 6) skor 0.483: oh theres the cheat in the place where tropical breezes flow the cheat


## 4. Uji kewarasan (bukan evaluasi resmi)
200 chunk dipilih acak; kueri dibuat dari teks chunk itu: *exact*, parsial (±30% kata dibuang), dan berisik (±8% huruf diganti).
Peringkat dihitung pada tingkat lagu (lagu muncul pertama kali = skor maksimum chunk-nya).
Evaluasi resmi dilakukan di notebook 06 dengan kueri dari notebook 05.

In [5]:
rng = np.random.default_rng(42)
cand = np.flatnonzero(chunks["n_songs_same_text"].values == 1)
pick = rng.choice(cand, size=min(200, len(cand)), replace=False)

def make_variants(text, rng):
    w = text.split()
    keep = [x for x in w if rng.random() > 0.3] or w[:2]
    chars = list(text)
    for i, ch in enumerate(chars):
        if ch.isalpha() and rng.random() < 0.08:
            chars[i] = str(rng.choice(list("abcdefghijklmnopqrstuvwxyz")))
    return {"exact": text, "partial": " ".join(keep), "noisy": "".join(chars)}

def song_rank(idx, target):
    seen = []
    for c in idx:
        s = song_of[c]
        if s not in seen:
            seen.append(s)
        if len(seen) == 10:
            break
    return seen.index(target) + 1 if target in seen else None

res = []
for name, fn in [("BM25", search_bm25), ("TF-IDF karakter", search_tfidf)]:
    for var in ["exact", "partial", "noisy"]:
        rr, t0 = [], time.time()
        for c in pick:
            q = make_variants(texts[c], rng)[var]
            r = song_rank(fn(q)[0], song_of[c])
            rr.append(0.0 if r is None else 1.0 / r)
        res.append((name, var, np.mean([x > 0 for x in rr]), np.mean(rr),
                    (time.time() - t0) / len(pick) * 1000))
display(pd.DataFrame(res, columns=["retriever", "varian", "Recall@10", "MRR", "ms/kueri"]).round(3))

,retriever,varian,Recall@10,MRR,ms/kueri
0,BM25,exact,1.000,0.997,42.502
1,BM25,partial,1.000,0.993,35.018
2,BM25,noisy,0.995,0.985,36.961
3,TF-IDF karakter,exact,1.000,1.000,319.643
4,TF-IDF karakter,partial,0.995,0.988,241.173
5,TF-IDF karakter,noisy,1.000,1.000,281.052


## 5. Simpan indeks
Hanya disimpan jika `N_LIMIT = None` (indeks penuh).

In [6]:
if N_LIMIT is None:
    t0 = time.time()
    sp.save_npz(f"{OUT_DIR}/bm25_csc.npz", bm25, compressed=False)
    sp.save_npz(f"{OUT_DIR}/tfidf_csc.npz", tfidf, compressed=False)
    np.save(f"{OUT_DIR}/tfidf_idf.npy", idf_c)
    with open(f"{OUT_DIR}/bm25_vocab.json", "w") as f:
        json.dump({k: int(v) for k, v in vocab.items()}, f)
    with open(f"{OUT_DIR}/lexical_params.json", "w") as f:
        json.dump({"n_chunks": N, "k1": K1, "b": B, "avgdl": avgdl, "ngram": list(NGRAM),
                   "hash_features": HASH_FEATURES, "sublinear_tf": SUBLINEAR_TF,
                   "chunk_file": os.path.basename(path)}, f, indent=2)
    for fn in sorted(os.listdir(OUT_DIR)):
        print(f"{fn:25s} {os.path.getsize(os.path.join(OUT_DIR, fn))/1e6:9.1f} MB")
    print(f"{time.time()-t0:.0f} dtk")
else:
    print("N_LIMIT masih aktif: indeks tidak disimpan.")

__notebook__.ipynb              0.0 MB
bm25_csc.npz                  254.5 MB
bm25_vocab.json                 2.9 MB
lexical_params.json             0.0 MB
tfidf_csc.npz                3717.8 MB
tfidf_idf.npy                  16.8 MB
4 dtk
